# Connect-4 Tournament Play — MCTS + Soft Actor-Critic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stiles-Clements1/connect4-rl-arena/blob/re_org/notebooks/tournament_play.ipynb)

Live-match harness for tournament play. Our agent uses Monte-Carlo
tree search guided by the Soft Actor-Critic policy + Q heads (the same
SAC model in `RL models/soft_actor_critic.keras` you've been
evaluating); we just run a configurable number of simulations per move
and take the most-visited child of the root, so the agent can see
multi-ply tactics and double-threat positions that the raw policy
misses.

**How to use during tournament:**
1. Run Cell 1 (loads the model, builds the MCTS agent, **warms up
   `@tf.function`** so the first real move isn't slow).
2. Run Cell 2 (game logic).
3. Run Cell 3 to start a game where we go first, or Cell 4 to start a
   game where the opponent goes first.
4. After each of our moves the cell prints the column to play and the
   wall-clock time it took. After each opponent move you type the
   column they played (1-7) and hit Enter.

**Strength vs. time trade-off** is controlled by `N_SIMULATIONS` in
Cell 1. Higher = stronger but slower. Rough rule of thumb on Colab T4
GPU at the default `N_PARALLEL_SIMS=16`:

| N_SIMULATIONS | est. time/move |
|---|---|
| 200 | ~0.4 s |
| 400 | ~0.8 s |
| 800 | ~1.5 s |
| **1000 (default)** | **~1.9 s** |
| 1600 | ~3.0 s |

Tune to fit whatever per-move time budget the tournament gives you.


In [ ]:
# Cell 1 — setup, load SAC, build MCTSAgent.

# ── CONFIG ──────────────────────────────────────────────────────────────────
# Strength / speed knobs. Tune these for your tournament time budget.
N_SIMULATIONS    = 1000   # MCTS rollouts per agent move (higher = stronger, slower)
N_PARALLEL_SIMS  = 16     # leaves evaluated per batched forward pass (8-32)
C_PUCT           = 1.4    # exploration constant; 1.0-2.0 range is standard
VALUE_METHOD     = 'mean_q'  # 'mean_q' (uses SAC Q heads) or 'rollout' (random playouts)
USE_TACTICS      = True   # immediate-win / immediate-block before MCTS
ADD_ROOT_NOISE   = False  # Dirichlet noise at the root; OFF for tournament play

# Random-opening knob. For the first N moves of the game (where N is drawn
# from a right-tailed distribution with mean RANDOM_OPENING_MEAN, clipped
# to [RANDOM_OPENING_MIN, RANDOM_OPENING_MAX]), the agent plays a uniformly
# random legal column instead of running MCTS. After that the agent
# switches to full-strength MCTS for the rest of the game.
#
# The point: MCTS at deterministic argmax always plays the same opening
# from the empty board, so an opponent who's seen us play once can
# pre-counter every game. Picking a random N per game off a right-tailed
# distribution means even our random-opening LENGTH varies, which makes
# our play unpredictable for the first ~3-7 of OUR moves on average.
ENABLE_RANDOM_OPENING = True
RANDOM_OPENING_MIN    = 4
RANDOM_OPENING_MAX    = 14
RANDOM_OPENING_MEAN   = 6  # right-tailed (Poisson(MEAN-MIN) added to MIN, capped at MAX)

# Display knobs.
OUR_NAME         = 'Group 12'
OPP_NAME         = 'Opponent'
WE_GO_FIRST      = True   # default for play(); override per-game with play(we_go_first=False)
# ────────────────────────────────────────────────────────────────────────────

import os, sys, subprocess, time
from pathlib import Path

GITHUB_OWNER  = 'Stiles-Clements1'
GITHUB_REPO   = 'connect4-rl-arena'
GITHUB_BRANCH = 're_org'
GITHUB_URL    = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    REPO_ROOT = Path(f'/content/{GITHUB_REPO}')
    if not REPO_ROOT.exists():
        print(f'Cloning {GITHUB_URL} (branch {GITHUB_BRANCH}) → {REPO_ROOT}')
        subprocess.run(
            ['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_URL, str(REPO_ROOT)],
            check=True,
        )
    else:
        print(f'Repo already at {REPO_ROOT}; pulling latest {GITHUB_BRANCH}.')
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch',  '--quiet', 'origin'], check=False)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout','--quiet', GITHUB_BRANCH], check=False)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull',   '--quiet', 'origin', GITHUB_BRANCH], check=False)
else:
    here = Path.cwd().resolve()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / 'src' / 'game_engine.py').exists()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError(f'Could not find repo root from {here}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Force-reimport src.* so a freshly-pulled session picks up new code
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]

# Quiet TF down before importing it
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
_gpus = tf.config.list_physical_devices('GPU')
for _g in _gpus:
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception:
        pass
HARDWARE = f'GPU ({_gpus[0].name.split(":")[-1]})' if _gpus else 'CPU only'
print(f'Hardware = {HARDWARE}')
print(f'Mode     = {"Colab" if IS_COLAB else "local"}')
print(f'REPO_ROOT = {REPO_ROOT}')

import numpy as np
from src import model_loader as ml
from src import game_engine as ge
from src.mcts import MCTSAgent

# Load the SAC model. Auto-detect encoding from input shape:
#   (None, 6, 7, 2) -> 'B'   (None, 6, 7, 1) -> 'A'   (None, 42, 2) -> 'B_flat'
SAC_PATH = REPO_ROOT / 'RL models' / 'soft_actor_critic.keras'
keras_model = tf.keras.models.load_model(str(SAC_PATH), compile=False)
try:
    in_shape = tuple(keras_model.input.shape)
except (AttributeError, TypeError):
    in_shape = tuple(keras_model.inputs[0].shape)
trailing = in_shape[-3:] if len(in_shape) >= 3 else None
if trailing == (6, 7, 2):
    encoding = 'B'
elif trailing == (6, 7, 1):
    encoding = 'A'
elif in_shape[-2:] == (42, 2):
    encoding = 'B_flat'
else:
    raise ValueError(f'Unrecognised input shape {in_shape}')
sac_wrapper = ml.ModelWrapper(keras_model, encoding, name='SAC')
print(f'\nLoaded SAC: input shape {in_shape}, encoding={encoding}')

# Build the MCTS agent
mcts_agent = MCTSAgent(
    sac_wrapper,
    n_simulations=N_SIMULATIONS,
    c_puct=C_PUCT,
    value_method=VALUE_METHOD,
    n_parallel_sims=N_PARALLEL_SIMS,
    use_tactics=USE_TACTICS,
    add_root_noise=ADD_ROOT_NOISE,
)

print(f'\nAgent: {mcts_agent.name}')
print(f'  N_SIMULATIONS    = {N_SIMULATIONS}')
print(f'  N_PARALLEL_SIMS  = {N_PARALLEL_SIMS}')
print(f'  C_PUCT           = {C_PUCT}')
print(f'  VALUE_METHOD     = {VALUE_METHOD}')
print(f'  USE_TACTICS      = {USE_TACTICS}')

# Warm-up move. The first @tf.function call triggers a (slow) graph
# trace; doing it now means the first move of the actual game isn't
# the one that pays the trace cost.
print(f'\nWarming up @tf.function (first call traces; this takes a few seconds)...')
_warm_t0 = time.perf_counter()
_ = mcts_agent.select_move(np.zeros((6, 7), dtype=np.int8), 1)
print(f'  warm-up move took {time.perf_counter() - _warm_t0:.1f}s')

# A second move to measure post-warm-up speed (this is roughly what
# real moves will cost during the game).
_warm_t0 = time.perf_counter()
_ = mcts_agent.select_move(np.zeros((6, 7), dtype=np.int8), 1)
_post_warm_ms = (time.perf_counter() - _warm_t0) * 1000
print(f'  steady-state move took {_post_warm_ms:.0f}ms')
print(f'\nReady to play.')


In [ ]:
# Cell 2 — emoji board + agent move with timing.

RED    = '\U0001F534'  # 🔴
YELLOW = '\U0001F7E1'  # 🟡
EMPTY  = '\u26AA'      # ⚪

def sample_random_opening_budget():
    """Draw N for this game from a right-tailed distribution with
    mean ~RANDOM_OPENING_MEAN, clipped to [MIN, MAX].

    Uses Poisson(MEAN - MIN) + MIN, capped at MAX. Most values cluster
    just above MIN with a long thin right tail; the cap rarely fires.
    Returns 0 if random opening is disabled.
    """
    if not ENABLE_RANDOM_OPENING:
        return 0
    n = RANDOM_OPENING_MIN + int(np.random.poisson(RANDOM_OPENING_MEAN - RANDOM_OPENING_MIN))
    return min(RANDOM_OPENING_MAX, n)

def our_move(board, player, move_number, random_open_budget):
    """Pick our move. For the first `random_open_budget` of OUR moves,
    play uniformly random; afterward, run MCTS with timing."""
    if move_number < random_open_budget:
        legal = ge.legal_moves(board)
        col = int(np.random.choice(legal))
        print(f'  [random opening move {move_number + 1}/{random_open_budget} — col {col + 1}]')
        return col
    t0 = time.perf_counter()
    col = mcts_agent.select_move(board, player)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    print(f'  [agent took {elapsed_ms:.0f}ms with N_SIMULATIONS={N_SIMULATIONS}, '
          f'N_PARALLEL_SIMS={N_PARALLEL_SIMS}]')
    return col

def render(board, our_side, last_col=None, status=None):
    OUR_TOKEN = RED    if our_side ==  1 else YELLOW
    OPP_TOKEN = YELLOW if our_side ==  1 else RED
    print()
    print(f'  {OUR_TOKEN} {OUR_NAME}   vs   {OPP_NAME} {OPP_TOKEN}')
    print()
    print('   ' + '  '.join(str(c + 1) for c in range(7)))
    print('  ' + '─' * 20)
    for r in range(6):
        row = '  '
        for c in range(7):
            if   board[r, c] ==  1: row += RED    + ' '
            elif board[r, c] == -1: row += YELLOW + ' '
            else:                   row += EMPTY  + ' '
        print(row)
    print('  ' + '─' * 20)
    if last_col is not None:
        print(f'  Last move: column {last_col + 1}')
    if status:
        print(f'\n  {status}')
    print()

print('Game UI ready.')


In [ ]:
# Cell 3 — play a game. Run this cell to start; it'll prompt for input.

def play(we_go_first=WE_GO_FIRST):
    board    = np.zeros((6, 7), dtype=np.int8)
    our_side = 1 if we_go_first else -1
    current  = 1   # red always moves first in Connect-4
    last_col = None

    # Draw this game's random-opening budget (0 if the knob is OFF).
    random_open_budget = sample_random_opening_budget()
    if random_open_budget > 0:
        print(f'Random-opening budget for THIS game: '
              f'{random_open_budget} of our moves before MCTS kicks in.')
    our_moves_played = 0

    render(board, our_side)

    while True:
        legal = ge.legal_moves(board)
        if not legal:
            render(board, our_side, last_col=last_col, status='Draw — board full!')
            return

        if current == our_side:
            col = our_move(board, current, our_moves_played, random_open_budget)
            our_moves_played += 1
            print(f'  {OUR_NAME} plays column {col + 1} — place your piece, then press Enter')
            input('  (press Enter when done) ')
        else:
            while True:
                try:
                    col = int(input(f'  {OPP_NAME} played which column? (1-7): ')) - 1
                    if col in legal:
                        break
                    print('  That column is full or invalid, try again.')
                except ValueError:
                    print('  Enter a number 1–7.')

        board    = ge.make_move(board, col, current)
        last_col = col

        done, winner = ge.is_terminal(board)
        if done:
            if winner == 0:
                render(board, our_side, last_col=last_col, status='Draw!')
            else:
                w_name = OUR_NAME if winner == our_side else OPP_NAME
                render(board, our_side, last_col=last_col, status=f'{w_name} wins!')
            return

        current  = -current
        render(board, our_side, last_col=last_col)

play()


### Play with opponent going first

Run the cell below if the opponent has the first move in this game.

In [ ]:
play(we_go_first=False)
